# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata summary
md = dataset.metadata
print(f"{md.name}: {md.description}\n")
print(f"Published: {md.datePublished}")
print(f"Identifier: {md.identifier}")
print(f"License: {md.license}")
print(f"Keywords: {md.keywords}")

## 2. Data Overview

Review available **record sets**, their fields, and the corresponding `@id`s.

> All record set, field, and column references are performed using their Croissant `@id`s for reproducible access.

In [ ]:
# List all available record sets in the dataset using their @id
record_sets = dataset.record_sets()
print(f"Total number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'No name')}")
    print(f"  Description: {rs.get('description', 'No description')}")
    print(f"  Fields:")
    for field in rs.get('field', []):
        print(f"    - {field['@id']} ({field.get('name','unnamed')}) : type={field.get('dataType','unknown')}")
    print()

## 3. Data Extraction

Load data from each record set into a DataFrame for further analysis.

> All Croissant IDs used below (`record_set`, `field`) are referenced by their `@id`.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]

# Load each record set into a DataFrame (referenced by @id)
dataframes = dict()

for rsid in record_set_ids:
    print(f"Loading data for record set: {rsid}")
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")

# Choose the first record set for demonstration
if record_set_ids:
    main_record_set = record_set_ids[0]
    print(f"Default main record set selected: {main_record_set}")
    print(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)

Here we'll demonstrate basic data processing using `@id` references. For illustration, we:
- Select a numeric field using its `@id` (first float/integer field detected in the record set).
- Filter records where the numeric value exceeds a threshold.
- Normalize the filtered numeric field.
- Optionally group by a categorical field, if present.

All field names and processing are dynamic, using IDs.

In [ ]:
# --- Dynamic EDA using @id references ---
import numpy as np

# We'll use the main_record_set from above
df = dataframes[main_record_set]

In [ ]:
# Identify numeric (float/integer) fields by their Croissant @id
numeric_field_id = None
fields = [field for rs in dataset.record_sets() if rs['@id'] == main_record_set for field in rs.get('field',[])]
for field in fields:
    if field.get('dataType') in ['schema:Float', 'schema:Integer', 'schema:Number']:
        numeric_field_id = field['@id']
        break

if numeric_field_id is None:
    raise Exception("No numeric field found in the selected record set.")
else:
    print(f"Numeric field selected (by @id): {numeric_field_id}")

In [ ]:
# Check and convert the column to numeric, handling missing values
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = np.nanmean(df[numeric_field_id])
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > mean ({threshold:.2f}):")
print(filtered_df[[numeric_field_id]].head())

In [ ]:
# Normalize the numeric field for the filtered records
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

In [ ]:
# Try to find a categorical/groupable field (first field of type Text/String)
group_field_id = None
for field in fields:
    if field.get('dataType') in ['schema:Text', 'schema:String']:
        group_field_id = field['@id']
        break

if group_field_id and group_field_id in filtered_df:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id} (using @id):")
    print(grouped_df.head())
else:
    print("No suitable group (categorical) field found for grouping.")

## 5. Visualization

Visualize data distributions or field relationships.

> All visualizations refer to fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (by @id)
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='steelblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group_field_id is available, show boxplot grouped by it
if group_field_id and group_field_id in df:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

This notebook demonstrated how to programmatically explore the structure and contents of a Croissant-defined dataset, referencing all record sets and fields by their unique `@id`. You learned how to:

- Discover and reference entities by their `@id` with `mlcroissant`.
- Load tabular data from each record set.
- Identify numeric and categorical fields for analysis.
- Perform basic data filtering, normalization, grouping, and visualization—all referencing fields by their Croissant `@id`.

To go further, adapt the notebook to your needs, referencing `@id` throughout for reproducible, interoperable data processing!